# <font color="#418FDE" size="6.5" uppercase>**OLS Ridge**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Fit and interpret ordinary least-squares linear regression models. 
- Apply Ridge regularization for regression and classification tasks. 
- Select Ridge regularization strength using cross-validation. 


## **1. Least Squares Basics**

### **1.1. Assumptions Intuition**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_01_01.jpg?v=1783991277" width="250">



>* OLS fits the best straight-line approximation
>* Assumptions make interpretations and uncertainty trustworthy

>* Linearity summarizes average straight-line patterns
>* Independent observations prevent false model certainty

>* Check residuals for bias and uneven spread
>* Investigate outliers before changing the model



### **1.2. Using LinearRegression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_01_02.jpg?v=1783991280" width="250">



>* Fits straight-line predictions to continuous outcomes
>* Coefficients explain feature effects holding others constant

>* Separate data, fit, then inspect parameters
>* Interpret coefficient signs and feature scales

>* Coefficients show associations, not proven causes
>* Avoid extrapolation; interpret with context



In [ ]:
#@title Python Code - Using LinearRegression

# This example fits ordinary least squares regression.
# LinearRegression learns an intercept and coefficient.
# The plot compares data with fitted predictions.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import LinearRegression

# Create a small deterministic housing-style dataset.
area_sq_m = np.array([45, 55, 65, 75, 85, 95, 105, 115], dtype=float)
rent_dollars = np.array([980, 1120, 1260, 1410, 1540, 1690, 1810, 1960])

# LinearRegression expects a two-dimensional feature matrix.
X = area_sq_m.reshape(-1, 1)
y = rent_dollars

# Validate that each apartment has one matching rent value.
if X.shape[0] != y.shape[0]:
    raise ValueError("Each feature row needs one target value.")

# Fit ordinary least squares to learn the best line.
model = LinearRegression()
model.fit(X, y)

# Predict rents for the observed apartment sizes.
predicted_rent = model.predict(X)
r_squared = model.score(X, y)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Intercept: ${model.intercept_:.2f}")
print(f"Coefficient: ${model.coef_[0]:.2f} per square meter")
print(f"R-squared on this small dataset: {r_squared:.3f}")

# Plot the observed points and the fitted least-squares line.
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(area_sq_m, rent_dollars, label="Observed rents")
ax.plot(area_sq_m, predicted_rent, color="orange", label="Fitted line")
ax.set_title("Using LinearRegression for Ordinary Least Squares")
ax.set_xlabel("Apartment area (square meters)")
ax.set_ylabel("Monthly rent (dollars)")
ax.legend()
plt.show()



### **1.3. Residual Diagnostics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_01_03.jpg?v=1783991282" width="250">



>* Residuals show what the model misses
>* Patterns reveal poor fit or missing factors

>* Check residual plots for patterns
>* Investigate uneven spread and influential outliers

>* Check assumptions before interpreting OLS results
>* Use residual issues to improve models



In [ ]:
#@title Python Code - Residual Diagnostics

# This example checks residuals after linear regression.
# Residual patterns reveal missed nonlinear structure.
# The plot should show a curved pattern.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Create a small dataset with a curved true relationship.
rng = np.random.default_rng(42)
square_feet = np.linspace(500, 2500, 80)

# Rent rises faster for larger apartments than a line can capture.
noise = rng.normal(loc=0, scale=120, size=square_feet.size)
rent = 900 + 1.1 * square_feet + 0.00045 * square_feet**2 + noise

# Scikit-learn expects features as a two-dimensional array.
X = square_feet.reshape(-1, 1)
model = LinearRegression()

# Fit one ordinary least-squares regression model.
model.fit(X, rent)
fitted_rent = model.predict(X)
residuals = rent - fitted_rent

# Validate that predictions and residuals match the data length.
if residuals.shape[0] != rent.shape[0]:
    raise ValueError("Residual count must match observation count.")

# Print a few concise diagnostics for interpretation.
r2 = r2_score(rent, fitted_rent)
print(f"scikit-learn version: {sklearn.__version__}")
print(f"OLS slope: ${model.coef_[0]:.2f} per square foot")
print(f"R-squared: {r2:.3f}")
print("Curved residuals suggest the straight-line model is missing structure.")

# Plot residuals against fitted values to diagnose model fit.
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(fitted_rent, residuals, alpha=0.75, label="Residual")
ax.axhline(0, color="black", linewidth=1, label="Zero error")

# Labels make the diagnostic plot easier to read.
ax.set_title("Residual Diagnostics for an OLS Model")
ax.set_xlabel("Fitted rent in dollars")
ax.set_ylabel("Residual in dollars")
ax.legend()
plt.show()



## **2. Ridge Models**

### **2.1. Ridge Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_02_01.jpg?v=1783991274" width="250">



>* Stabilizes linear models with correlated predictors
>* Shrinks coefficients for better future predictions

>* Balances training fit with simpler coefficients
>* Shrinks coefficients while keeping predictors

>* Shrunken coefficients need careful interpretation
>* Improves prediction with correlated real-world predictors



In [ ]:
#@title Python Code - Ridge Regression

# This example compares ordinary and ridge regression.
# Ridge shrinks coefficients to improve prediction stability.
# Cross-validation chooses a useful regularization strength.

import numpy as np
import sklearn
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Correlated features make ordinary regression less stable.
rng = np.random.default_rng(42)
base_feature = rng.normal(size=160)

# Build several predictors carrying overlapping information.
X = np.column_stack(
    [base_feature + rng.normal(scale=0.08, size=160) for _ in range(8)]
)

# Add a target with signal plus random noise.
y = 12 * base_feature + rng.normal(scale=3.0, size=160)

# Validate the small teaching dataset before modeling.
if X.shape != (160, 8) or y.shape != (160,):
    raise ValueError("The generated regression data has an unexpected shape.")

# Split first, then fit scaling only on training data.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

# Standardization makes ridge penalties comparable across features.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Fit ordinary least squares without coefficient shrinkage.
ols_model = LinearRegression()
ols_model.fit(X_train_scaled, y_train)

# RidgeCV tests several alpha values using cross-validation.
alpha_values = np.array([0.01, 0.1, 1.0, 10.0, 100.0])
ridge_model = RidgeCV(alphas=alpha_values, cv=5)
ridge_model.fit(X_train_scaled, y_train)

# Compare test error and coefficient size.
ols_rmse = np.sqrt(mean_squared_error(y_test, ols_model.predict(X_test_scaled)))
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_model.predict(X_test_scaled)))

# Smaller coefficient norms show ridge shrinkage.
ols_norm = np.linalg.norm(ols_model.coef_)
ridge_norm = np.linalg.norm(ridge_model.coef_)

print(f"scikit-learn version: {sklearn.__version__}")
print(f"OLS test RMSE: {ols_rmse:.2f}")
print(f"RidgeCV test RMSE: {ridge_rmse:.2f}")
print(f"Chosen ridge alpha: {ridge_model.alpha_:.2f}")
print(f"OLS coefficient norm: {ols_norm:.2f}")
print(f"Ridge coefficient norm: {ridge_norm:.2f}")



### **2.2. Ridge Classification**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_02_02.jpg?v=1783991272" width="250">



>* Ridge classifiers predict categories, not numbers
>* Regularization shrinks coefficients for stable decisions

>* Ridge creates steadier class boundaries
>* Shared predictor influence reduces overfitting

>* Stable predictions using all available predictors
>* Balance shrinkage to preserve useful patterns



In [ ]:
#@title Python Code - Ridge Classification

# This example demonstrates Ridge classification.
# Regularization shrinks coefficients for stable decisions.
# Accuracy and coefficient size are compared.

import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Create a small classification dataset with related predictors.
features, target = make_classification(
    n_samples=600,
    n_features=12,
    n_informative=5,
    n_redundant=5,
    random_state=42,
)

# Check that the feature matrix matches the target vector.
if features.shape[0] != target.shape[0]:
    raise ValueError("Features and target must have matching rows.")

# Split data so evaluation uses unseen examples.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.30,
    stratify=target,
    random_state=42,
)

# Weak regularization allows larger coefficient values.
weak_ridge = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=100.0, penalty="l2", max_iter=1000, random_state=42),
)

# Stronger Ridge regularization shrinks coefficients more.
strong_ridge = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=0.1, penalty="l2", max_iter=1000, random_state=42),
)

# Fit both classifiers on the same training data.
weak_ridge.fit(X_train, y_train)
strong_ridge.fit(X_train, y_train)

# Compare accuracy and average absolute coefficient size.
weak_accuracy = accuracy_score(y_test, weak_ridge.predict(X_test))
strong_accuracy = accuracy_score(y_test, strong_ridge.predict(X_test))

# Extract the logistic regression step from each pipeline.
weak_model = weak_ridge.named_steps["logisticregression"]
strong_model = strong_ridge.named_steps["logisticregression"]

# Smaller average coefficients show stronger shrinkage.
weak_size = abs(weak_model.coef_).mean()
strong_size = abs(strong_model.coef_).mean()

print("scikit-learn version:", sklearn.__version__)
print("Weak Ridge accuracy:", round(weak_accuracy, 3))
print("Strong Ridge accuracy:", round(strong_accuracy, 3))
print("Weak Ridge mean coefficient size:", round(weak_size, 3))
print("Strong Ridge mean coefficient size:", round(strong_size, 3))



### **2.3. Scaling Effects**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_02_03.jpg?v=1783991276" width="250">



>* Ridge penalties depend on feature scale
>* Scaling makes shrinkage fair across predictors

>* Standardize features for balanced Ridge penalties
>* Scaling prevents unit-driven feature dominance

>* Standardization makes Ridge strength comparable
>* Apply training scaling consistently to all data



In [ ]:
#@title Python Code - Scaling Effects

# This example shows why Ridge models need scaling.
# Feature units can change regularization behavior unfairly.
# Standardization makes Ridge coefficients more comparable.

import numpy as np
import sklearn
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Create two equivalent features with very different numeric scales.
rng = np.random.default_rng(42)
area_square_meters = rng.uniform(50, 200, size=120)
area_square_centimeters = area_square_meters * 10000

# The target depends on area plus small random noise.
noise = rng.normal(0, 10000, size=120)
price_euros = 50000 + 3000 * area_square_meters + noise

# Combine the same information recorded in two different units.
features = np.column_stack([area_square_meters, area_square_centimeters])
if features.shape != (120, 2):
    raise ValueError("Expected 120 rows and 2 feature columns.")

# Fit Ridge directly on unscaled features.
unscaled_ridge = Ridge(alpha=1000.0)
unscaled_ridge.fit(features, price_euros)
unscaled_score = unscaled_ridge.score(features, price_euros)

# Fit Ridge after learning scaling from the training data.
scaled_ridge = make_pipeline(StandardScaler(), Ridge(alpha=1000.0))
scaled_ridge.fit(features, price_euros)
scaled_score = scaled_ridge.score(features, price_euros)

# Extract the Ridge coefficients after standardization.
scaled_step = scaled_ridge.named_steps["ridge"]
scaled_coefficients = scaled_step.coef_

print("scikit-learn version:", sklearn.__version__)
print("Unscaled coefficients:", np.round(unscaled_ridge.coef_, 6))
print("Scaled Ridge coefficients:", np.round(scaled_coefficients, 2))
print("Unscaled training R^2:", round(unscaled_score, 3))
print("Scaled training R^2:", round(scaled_score, 3))
print("Lesson: scaling makes the Ridge penalty treat units more fairly.")



## **3. Ridge Selection**

### **3.1. Ridge Cross Validation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_03_01.jpg?v=1783991267" width="250">



>* Ridge shrinkage stabilizes noisy regression models
>* Cross validation chooses the best shrinkage

>* Train and test Ridge on data splits
>* Choose shrinkage that predicts held-out data best

>* Choose shrinkage using validation evidence
>* Refit carefully with realistic validation splits



In [ ]:
#@title Python Code - Ridge Cross Validation

# This example selects Ridge strength using cross-validation.
# Correlated predictors make regularization choices important.
# The plot shows validation error across alphas.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_regression
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# Create a small regression dataset with correlated predictors.
features, target = make_regression(
    n_samples=220,
    n_features=12,
    n_informative=5,
    noise=25.0,
    random_state=42,
)

# Add correlation by blending neighboring feature columns.
features[:, 6:] = features[:, :6] + 0.15 * features[:, 6:]

# Check that the generated data has matching rows.
if features.shape[0] != target.shape[0]:
    raise ValueError("Features and target must have matching rows.")

# Hold out test data for a final unbiased check.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.25,
    random_state=42,
)

# Try several Ridge strengths from weak to strong shrinkage.
alphas = np.array([0.01, 0.1, 1.0, 10.0, 100.0])

# RidgeCV chooses the alpha with best average validation performance.
ridge_cv = make_pipeline(
    StandardScaler(),
    RidgeCV(alphas=alphas, cv=5, scoring="neg_mean_squared_error"),
)

# Fit preprocessing and Ridge only on the training data.
ridge_cv.fit(X_train, y_train)

# Extract the selected Ridge model from the pipeline.
selected_ridge = ridge_cv.named_steps["ridgecv"]
selected_alpha = selected_ridge.alpha_

# Evaluate the final selected model on held-out test data.
test_predictions = ridge_cv.predict(X_test)
test_rmse = mean_squared_error(y_test, test_predictions) ** 0.5

# Convert cross-validation scores into easier RMSE values.
mean_cv_mse = np.array([
    -cross_val_score(
        make_pipeline(StandardScaler(), Ridge(alpha=alpha)),
        X_train,
        y_train,
        cv=5,
        scoring="neg_mean_squared_error",
    ).mean()
    for alpha in alphas
])
mean_cv_rmse = mean_cv_mse ** 0.5

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Selected alpha: {selected_alpha}")
print(f"Test RMSE after RidgeCV selection: {test_rmse:.2f}")

# Plot validation error for each candidate alpha.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(alphas, mean_cv_rmse, marker="o", label="5-fold CV RMSE")
ax.axvline(selected_alpha, color="red", linestyle="--", label="selected alpha")
ax.set_xscale("log")
ax.set_title("Ridge Cross Validation Selects Alpha")
ax.set_xlabel("Ridge alpha, larger means more shrinkage")
ax.set_ylabel("Average validation RMSE")
ax.legend()
plt.show()



### **3.2. Multioutput Ridge**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_03_02.jpg?v=1783991268" width="250">



>* Predict multiple related outcomes from shared features
>* Cross-validate regularization across all outputs

>* Cross-validate alpha across all target outputs
>* Balance metrics for scale and importance

>* Stabilizes related outputs with shared predictors
>* Balance regularization to avoid noise or oversimplification



In [ ]:
#@title Python Code - Multioutput Ridge

# This example selects Ridge strength for multiple outputs.
# Cross-validation compares alphas using shared input features.
# The result shows balanced multioutput test performance.

import numpy as np
import sklearn
from sklearn.datasets import make_regression
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Generate a small deterministic multioutput regression dataset.
features, targets = make_regression(
    n_samples=180,
    n_features=8,
    n_targets=3,
    noise=18.0,
    random_state=42,
)

# Give outputs different scales to motivate target standardization.
target_scales = np.array([1.0, 10.0, 0.2])
targets = targets * target_scales

# Check that the feature and target rows match.
if features.shape[0] != targets.shape[0]:
    raise ValueError("Features and targets must have matching rows.")

# Split before scaling to avoid test data leakage.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    targets,
    test_size=0.25,
    random_state=42,
)

# Standardize features and outputs using training data only.
feature_scaler = StandardScaler()
target_scaler = StandardScaler()
X_train_scaled = feature_scaler.fit_transform(X_train)
X_test_scaled = feature_scaler.transform(X_test)

# Scaling targets makes each output count more evenly.
y_train_scaled = target_scaler.fit_transform(y_train)
y_test_scaled = target_scaler.transform(y_test)

# RidgeCV chooses alpha by cross-validation across all outputs.
candidate_alphas = np.array([0.01, 0.1, 1.0, 10.0, 100.0])
ridge_cv = RidgeCV(alphas=candidate_alphas, cv=5)
ridge_cv.fit(X_train_scaled, y_train_scaled)

# Evaluate the selected model on held-out test data.
y_pred_scaled = ridge_cv.predict(X_test_scaled)
per_output_mse = mean_squared_error(
    y_test_scaled,
    y_pred_scaled,
    multioutput="raw_values",
)

# Print concise results for the multioutput selection lesson.
print("scikit-learn version:", sklearn.__version__)
print("Candidate alphas:", candidate_alphas.tolist())
print("Selected alpha:", float(ridge_cv.alpha_))
print("Test MSE by standardized output:", np.round(per_output_mse, 3).tolist())
print("Average standardized test MSE:", round(float(per_output_mse.mean()), 3))



### **3.3. Interpreting Ridge Coefficients**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_09/Lecture_A/image_03_03.jpg?v=1783991270" width="250">



>* Ridge coefficients show adjusted prediction changes
>* Cross-validation chooses useful shrinkage for prediction

>* Ridge stabilizes correlated predictor coefficients
>* Interpret related features as groups

>* Cross-validation chooses the best shrinkage level
>* Standardize features before comparing Ridge coefficients



In [ ]:
#@title Python Code - Interpreting Ridge Coefficients

# This example compares Ridge coefficients after cross-validation.
# Standardization makes coefficient sizes easier to compare.
# The selected alpha shows useful shrinkage for prediction.

import numpy as np
import pandas as pd
import sklearn
from sklearn.datasets import make_regression

from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Correlated features make ordinary coefficients less stable.
rng = np.random.default_rng(42)
base_size = rng.normal(0, 1, 120)

noise = rng.normal(0, 0.15, 120)
square_feet = base_size + noise
bedrooms = base_size + rng.normal(0, 0.15, 120)

age = rng.normal(0, 1, 120)
distance = rng.normal(0, 1, 120)

# The target uses a clear positive size effect.
price = 80 * square_feet + 20 * bedrooms - 35 * age - 10 * distance
price = price + rng.normal(0, 25, 120)

feature_names = ["square_feet", "bedrooms", "age", "distance"]
X = np.column_stack([square_feet, bedrooms, age, distance])
y = price

# Validate the small teaching dataset before modeling.
if X.shape != (120, 4) or y.shape != (120,):
    raise ValueError("The teaching data has an unexpected shape.")

if not np.isfinite(X).all() or not np.isfinite(y).all():
    raise ValueError("The teaching data must contain only finite values.")

# RidgeCV tests several shrinkage strengths using cross-validation.
alphas = [0.01, 0.1, 1.0, 10.0, 100.0]
model = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas, cv=5))

model.fit(X, y)
ridge_step = model.named_steps["ridgecv"]

# Coefficients are on standardized feature scales.
coef_table = pd.DataFrame(
    {"feature": feature_names, "ridge_coefficient": ridge_step.coef_}
)

coef_table["ridge_coefficient"] = coef_table["ridge_coefficient"].round(2)
coef_table = coef_table.sort_values("ridge_coefficient", ascending=False)

print("scikit-learn version:", sklearn.__version__)
print("Selected alpha from 5-fold CV:", ridge_step.alpha_)
print("Ridge coefficients after standardization:")
print(coef_table.to_string(index=False))



# <font color="#418FDE" size="6.5" uppercase>**OLS Ridge**</font>


In this lecture, you learned to:
- Fit and interpret ordinary least-squares linear regression models. 
- Apply Ridge regularization for regression and classification tasks. 
- Select Ridge regularization strength using cross-validation. 

In the next Lecture (Lecture B), we will go over 'Lasso Elastic Net'